# Register the TimesFM champion

TimesFM (zero-shot) is the best model on the hidden test set (Kaggle public 2,969 /
private 3,130), beating the ensemble (3,923). This notebook wraps it as an MLflow
**pyfunc** that takes the **raw** `test.csv` and returns forecasts, then registers it as
`Walmart_Champion_TimesFM`.

The wrapper bundles the sales history and re-loads the pretrained TimesFM checkpoint on
load, so it runs on raw test data directly.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

cwd: D:\Desktop\ML Project


In [2]:
import warnings
import numpy as np
import pandas as pd
import mlflow
import mlflow.pyfunc
import dagshub

from src.features.nn_preprocessing import build_long_df

warnings.filterwarnings("ignore")
dagshub.init(repo_owner="slosa23", repo_name="Walmart-Sales-Forecasting", mlflow=True)
mlflow.set_experiment("Champion_Registry")
print("Tracking URI:", mlflow.get_tracking_uri())

D:\Desktop\ML Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Accessing as GigiSichinava

Initialized MLflow to track repo "slosa23/Walmart-Sales-Forecasting"

Repository slosa23/Walmart-Sales-Forecasting initialized!

Tracking URI: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow


In [3]:
train_raw = pd.read_csv("data/train.csv")
stores = pd.read_csv("data/stores.csv")
features = pd.read_csv("data/features.csv")

os.makedirs("champion_artifacts", exist_ok=True)
history = build_long_df(train_raw, stores, features, fill_gaps=True)[["unique_id", "ds", "y"]]
history.to_parquet("champion_artifacts/history_timesfm.parquet")
print("history rows:", len(history))

history rows: 449237


## The TimesFM champion pyfunc (runs on raw test.csv)

In [4]:
class TimesFMChampion(mlflow.pyfunc.PythonModel):
    FREQ = "W-FRI"

    def load_context(self, context):
        import pandas as pd
        from timesfm.timesfm_2p5.timesfm_2p5_torch import TimesFM_2p5_200M_torch
        self.history = pd.read_parquet(context.artifacts["history"])
        self.model = TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
        self._compiled_h = None

    def predict(self, context, model_input):
        import pandas as pd, numpy as np
        from timesfm import ForecastConfig
        test = model_input.copy()
        test["ds"] = pd.to_datetime(test["Date"])
        horizon = int(test["ds"].nunique())
        if self._compiled_h != horizon:
            self.model.compile(ForecastConfig(max_context=512, max_horizon=horizon,
                                              normalize_inputs=True, infer_is_positive=True,
                                              per_core_batch_size=32))
            self._compiled_h = horizon

        uids, series_list, last_ds = [], [], {}
        for uid, g in self.history.groupby("unique_id", sort=False):
            g = g.sort_values("ds")
            uids.append(uid)
            series_list.append(g["y"].values.astype(float))
            last_ds[uid] = g["ds"].max()

        pf, _ = self.model.forecast(horizon=horizon, inputs=series_list)
        pf = np.asarray(pf)
        rows = []
        for i, uid in enumerate(uids):
            dates = pd.date_range(last_ds[uid] + pd.Timedelta(weeks=1), periods=horizon, freq=self.FREQ)
            for h in range(horizon):
                rows.append((uid, dates[h], float(pf[i][h])))
        fcst = pd.DataFrame(rows, columns=["unique_id", "ds", "timesfm"])

        test["unique_id"] = test["Store"].astype(str) + "_" + test["Dept"].astype(str)
        out = test.merge(fcst, on=["unique_id", "ds"], how="left")
        return out["timesfm"].fillna(0).clip(lower=0).values

D:\Desktop\ML Project\.venv\Lib\site-packages\mlflow\pyfunc\utils\data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


## Log + Register

In [5]:
with mlflow.start_run(run_name="Register_Champion_TimesFM"):
    mlflow.log_param("model", "timesfm-2.5-200m (zero-shot)")
    mlflow.log_metric("kaggle_public", 2969.44)
    info = mlflow.pyfunc.log_model(
        name="model",
        python_model=TimesFMChampion(),
        artifacts={"history": "champion_artifacts/history_timesfm.parquet"},
        pip_requirements=["timesfm[torch]", "torch", "pandas", "numpy", "pyarrow"],
        registered_model_name="Walmart_Champion_TimesFM",
    )
    print("Registered:", info.model_uri)

2026/07/12 08:05:01 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
Successfully registered model 'Walmart_Champion_TimesFM'.
2026/07/12 08:05:09 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Walmart_Champion_TimesFM, version 1
Created version '1' of model 'Walmart_Champion_TimesFM'.


Registered: models:/m-d87a09c7a94d4758ae96da8bb8e24671
🏃 View run Register_Champion_TimesFM at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/11/runs/7aae85e93143469b80ba17f61a1c338d
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/11
